# Construct Kuhn Poker

In [1]:
from pprint import pprint

Helper functions & variables

In [2]:
CARDS = ("J", "Q", "K")
RANK = {"J": 0, "Q": 1, "K": 2}
DEALS = ["KQ", "QK", "JK", "KJ", "QJ", "JQ"]


def showdown_utility(card0: str, card1: str, amount: int) -> list[int]:
    rank = {"J": 0, "Q": 1, "K": 2}
    u0 = amount if rank[card0] > rank[card1] else -amount
    return [u0, -u0]

1. Extensive-Form Game (EFG) form

In [3]:
def kuhn_efg() -> dict:
    efg = {
        "": {
            "type": "CHANCE",
            "outcomes": [],
        }
    }

    for deal in DEALS:
        card0, card1 = deal
        efg[""]["outcomes"].append((deal, deal, 1 / 6))

        # Player 0 first action (check / bet)
        efg[deal] = {
            "type": "DECISION",
            "player": 0,
            "information_set": card0,
            "actions": [("check", f"{deal}|check"), ("bet", f"{deal}|bet")],
        }

        # After check, player 1 can check or bet
        efg[f"{deal}|check"] = {
            "type": "DECISION",
            "player": 1,
            "information_set": f"{card1}|check",
            "actions": [
                ("check", f"{deal}|check-check"),
                ("bet", f"{deal}|check-bet"),
            ],
        }

        # After check-bet, player 0 can call or fold
        efg[f"{deal}|check-bet"] = {
            "type": "DECISION",
            "player": 0,
            "information_set": f"{card0}|check-bet",
            "actions": [
                ("call", f"{deal}|check-bet-call"),
                ("fold", f"{deal}|check-bet-fold"),
            ],
        }

        # After bet, player 1 can call or fold
        efg[f"{deal}|bet"] = {
            "type": "DECISION",
            "player": 1,
            "information_set": f"{card1}|bet",
            "actions": [("call", f"{deal}|bet-call"), ("fold", f"{deal}|bet-fold")],
        }

        # Terminal nodes
        efg[f"{deal}|check-check"] = {
            "type": "TERMINAL",
            "utility": showdown_utility(card0, card1, amount=1),
        }
        efg[f"{deal}|check-bet-call"] = {
            "type": "TERMINAL",
            "utility": showdown_utility(card0, card1, amount=2),
        }
        efg[f"{deal}|check-bet-fold"] = {
            "type": "TERMINAL",
            "utility": [-1, 1],
        }
        efg[f"{deal}|bet-call"] = {
            "type": "TERMINAL",
            "utility": showdown_utility(card0, card1, amount=2),
        }
        efg[f"{deal}|bet-fold"] = {
            "type": "TERMINAL",
            "utility": [1, -1],
        }

    return efg


efg = kuhn_efg()
pprint(efg[''])
pprint(efg['KQ'])
pprint(efg['KQ|check'])

{'outcomes': [('KQ', 'KQ', 0.16666666666666666),
              ('QK', 'QK', 0.16666666666666666),
              ('JK', 'JK', 0.16666666666666666),
              ('KJ', 'KJ', 0.16666666666666666),
              ('QJ', 'QJ', 0.16666666666666666),
              ('JQ', 'JQ', 0.16666666666666666)],
 'type': 'CHANCE'}
{'actions': [('check', 'KQ|check'), ('bet', 'KQ|bet')],
 'information_set': 'K',
 'player': 0,
 'type': 'DECISION'}
{'actions': [('check', 'KQ|check-check'), ('bet', 'KQ|check-bet')],
 'information_set': 'Q|check',
 'player': 1,
 'type': 'DECISION'}


2. Tree-Form Sequential Decision Process (TFSDP) form

In [4]:
def kuhn_tfsdp(player: int) -> dict:
    if player == 0:
        J = [*CARDS, *[f"{c}|check-bet" for c in CARDS]]
        A = {c: ["check", "bet"] for c in CARDS}
        A.update({f"{c}|check-bet": ["call", "fold"] for c in CARDS})
        p = {c: None for c in CARDS}
        p.update({f"{c}|check-bet": (c, "check") for c in CARDS})

        K = ["", *[f"{c}|check" for c in CARDS]]
        S = {
            "": list(CARDS),
            **{f"{c}|check": ["check", "bet"] for c in CARDS},
        }

        rho = {}
        for c in CARDS:
            rho[("", c)] = c
            rho[(c, "check")] = f"{c}|check"
            rho[(c, "bet")] = "T"
            rho[(f"{c}|check", "check")] = "T"
            rho[(f"{c}|check", "bet")] = f"{c}|check-bet"
            rho[(f"{c}|check-bet", "call")] = "T"
            rho[(f"{c}|check-bet", "fold")] = "T"

    else:
        J = [f"{c}|check" for c in CARDS] + [f"{c}|bet" for c in CARDS]
        A = {f"{c}|check": ["check", "bet"] for c in CARDS}
        A.update({f"{c}|bet": ["call", "fold"] for c in CARDS})
        p = {j: None for j in J}

        K = ["", *CARDS]
        S = {
            "": list(CARDS),
            **{c: ["check", "bet"] for c in CARDS},
        }

        rho = {}
        for c in CARDS:
            rho[("", c)] = c
            rho[(c, "check")] = f"{c}|check"
            rho[(c, "bet")] = f"{c}|bet"
            rho[(f"{c}|check", "check")] = "T"
            rho[(f"{c}|check", "bet")] = "T"
            rho[(f"{c}|bet", "call")] = "T"
            rho[(f"{c}|bet", "fold")] = "T"

    Sigma = [(j, a) for j in J for a in A[j]]

    return {
        "J": J,
        "A": A,
        "K": K,
        "S": S,
        "rho": rho,
        "Sigma": Sigma,
        "p": p,
    }


tfsdp0 = kuhn_tfsdp(0)
tfsdp1 = kuhn_tfsdp(1)

pprint(tfsdp0['J'])
pprint(tfsdp0['A'])
pprint(tfsdp0['K'])
pprint(tfsdp0['S'])
# pprint(tfsdp0['rho'])
# pprint(tfsdp0['Sigma'])
# pprint(tfsdp0['p'])

['J', 'Q', 'K', 'J|check-bet', 'Q|check-bet', 'K|check-bet']
{'J': ['check', 'bet'],
 'J|check-bet': ['call', 'fold'],
 'K': ['check', 'bet'],
 'K|check-bet': ['call', 'fold'],
 'Q': ['check', 'bet'],
 'Q|check-bet': ['call', 'fold']}
['', 'J|check', 'Q|check', 'K|check']
{'': ['J', 'Q', 'K'],
 'J|check': ['check', 'bet'],
 'K|check': ['check', 'bet'],
 'Q|check': ['check', 'bet']}
